# 12_multi_head_attention

Audience: junior researcher

- Challenge path: `challenges/hard/12_multi_head_attention`
- Source spec: [challenges/hard/12_multi_head_attention/challenge.html](../challenge.html)
- Source implementation: [challenges/hard/12_multi_head_attention/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Implement a program for multi-head self-attention. Given three input matrices \(Q\) (queries), \(K\) (keys), and \(V\) (values) of size \(N \times d_{\text{model}}\), compute:
  \[ \text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,\ldots,\text{head}_h) \]
  where each head computes:
  \[ \text{head}_i = \text{softmax}\left(\frac{Q_iK_i^T}{\sqrt{d_k}}\right)V_i \]
  with \(d_k = d_{\text{model}}/h\) and \(Q_i, K_i, V_i\) being the i-th head's partition of the input matrices.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> array</li>
</ul>

<h2>Example 1:</h2>
<p>
Input:
\[
\begin{align*}
N &= 2, \quad d_{\text{model}} = 4, \quad h = 2 \\[1em]
Q &= \begin{bmatrix}
1.0 & 0.0 & 2.0 & 3.0 \\
4.0 & 5.0 & 6.0 & 7.0
\end{bmatrix} \\[1em]
K &= \begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0
\end{bmatrix} \\[1em]
V &= \begin{bmatrix}
0.5 & 1.0 & 1.5 & 2.0 \\
2.5 & 3.0 & 3.5 & 4.0
\end{bmatrix}
\end{align*}
\]

Output:
\[
\begin{bmatrix}
2.39 & 2.89 & 3.50 & 4.00 \\
2.50 & 3.00 & 3.50 & 4.00
\end{bmatrix}
\]
</p>

<h2>Example 2:</h2>
<p>
Input:
\[
\begin{align*}
N &= 1, \quad d_{\text{model}} = 2, \quad h = 1 \\[1em]
Q &= \begin{bmatrix} 1.0 & 1.0 \end{bmatrix} \\[1em]
K &= \begin{bmatrix} 1.0 & 1.0 \end{bmatrix} \\[1em]
V &= \begin{bmatrix} 2.0 & 3.0 \end{bmatrix}
\end{align*}
\]

Output:
\[
\begin{bmatrix} 2.0 & 3.0 \end{bmatrix}
\]
</p>

<h2>Constraints</h2>
<ul>
  <li><code>1 ≤ N ≤ 10000</code></li>
  <li><code>2 ≤ d_model ≤ 1024</code></li>
  <li><code>1 ≤ h ≤ d_model</code></li>
  <li><code>d_model % h == 0</code></li>
  <li><code>-10.0 ≤ values ≤ 10.0</code></li>

  <li>Performance is measured with <code>N</code> = 1,024, <code>d_model</code> = 1,024</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/hard/12_multi_head_attention/solution/solution.pytorch.py`

In [ ]:
import math

import torch


def solve(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, output: torch.Tensor, N: int, d_model: int, h: int):
    d_k = d_model // h
    scale = math.sqrt(d_k)
    pieces = []
    for head in range(h):
        start = head * d_k
        stop = start + d_k
        q_h = Q[:, start:stop]
        k_h = K[:, start:stop]
        v_h = V[:, start:stop]
        scores = torch.matmul(q_h, k_h.t()) / scale
        weights = torch.softmax(scores, dim=-1)
        pieces.append(torch.matmul(weights, v_h))
    output.copy_(torch.cat(pieces, dim=-1))


## Jax

Source: `challenges/hard/12_multi_head_attention/solution/solution.jax.py`

In [ ]:
from __future__ import annotations

from typing import Any
import math

import torch
import torch.nn.functional as F

try:
    import jax
    import jax.numpy as jnp
except Exception:  # pragma: no cover - optional dependency
    jax = None
    jnp = None


def _torch_impl(Q: Any, K: Any, V: Any, N: int, d_model: int, h: int):
    d_k = d_model // h
    scale = math.sqrt(d_k)
    pieces = []
    for head in range(h):
        start = head * d_k
        stop = start + d_k
        q_h = Q[:, start:stop]
        k_h = K[:, start:stop]
        v_h = V[:, start:stop]
        scores = torch.matmul(q_h, k_h.t()) / scale
        weights = torch.softmax(scores, dim=-1)
        pieces.append(torch.matmul(weights, v_h))
    return torch.cat(pieces, dim=-1)


def _jax_impl(Q: Any, K: Any, V: Any, N: int, d_model: int, h: int):
    d_k = d_model // h
    scale = math.sqrt(d_k)
    pieces = []
    for head in range(h):
        start = head * d_k
        stop = start + d_k
        q_h = Q[:, start:stop]
        k_h = K[:, start:stop]
        v_h = V[:, start:stop]
        scores = jnp.matmul(q_h, k_h.T) / scale
        scores = scores - jnp.max(scores, axis=-1, keepdims=True)
        weights = jnp.exp(scores)
        weights = weights / jnp.sum(weights, axis=-1, keepdims=True)
        pieces.append(jnp.matmul(weights, v_h))
    return jnp.concatenate(pieces, axis=-1)


if jax is None:

    def solve(Q: Any, K: Any, V: Any, N: int, d_model: int, h: int):
        return _torch_impl(Q, K, V, N, d_model, h)

else:
    solve = jax.jit(_jax_impl, static_argnames=("N", "d_model", "h"))


## Triton

Source: `challenges/hard/12_multi_head_attention/solution/solution.triton.py`

In [ ]:
import math

import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover - optional dependency
    triton = None
    tl = None


def solve(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, output: torch.Tensor, N: int, d_model: int, h: int):
    d_k = d_model // h
    scale = math.sqrt(d_k)
    pieces = []
    for head in range(h):
        start = head * d_k
        stop = start + d_k
        q_h = Q[:, start:stop]
        k_h = K[:, start:stop]
        v_h = V[:, start:stop]
        scores = torch.matmul(q_h, k_h.t()) / scale
        weights = torch.softmax(scores, dim=-1)
        pieces.append(torch.matmul(weights, v_h))
    output.copy_(torch.cat(pieces, dim=-1))


## Mlx

Source: `challenges/hard/12_multi_head_attention/solution/solution.mlx.py`

In [ ]:
from __future__ import annotations

from typing import Any
import importlib.util
import math

import torch


def _torch_impl(Q: Any, K: Any, V: Any, output: Any, N: int, d_model: int, h: int):
    d_k = d_model // h
    scale = math.sqrt(d_k)
    pieces = []
    for head in range(h):
        start = head * d_k
        stop = start + d_k
        q_h = Q[:, start:stop]
        k_h = K[:, start:stop]
        v_h = V[:, start:stop]
        scores = torch.matmul(q_h, k_h.t()) / scale
        weights = torch.softmax(scores, dim=-1)
        pieces.append(torch.matmul(weights, v_h))
    output.copy_(torch.cat(pieces, dim=-1))
    return output


def solve(Q: Any, K: Any, V: Any, output: Any, N: int, d_model: int, h: int):
    if importlib.util.find_spec("mlx.core") is None:
        return _torch_impl(Q, K, V, output, N, d_model, h)

    import mlx.core as mx

    d_k = d_model // h
    scale = math.sqrt(d_k)
    pieces = []
    for head in range(h):
        start = head * d_k
        stop = start + d_k
        q_h = Q[:, start:stop]
        k_h = K[:, start:stop]
        v_h = V[:, start:stop]
        scores = (q_h @ k_h.T) / scale
        scores = scores - mx.max(scores, axis=-1, keepdims=True)
        weights = mx.exp(scores)
        weights = weights / mx.sum(weights, axis=-1, keepdims=True)
        pieces.append(weights @ v_h)
    result = mx.concatenate(pieces, axis=-1)
    output[...] = result
    return output


## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.